# One-step retrosynthesis evaluation with vLLM

Self-contained vLLM port of the generation loop in `code_v1.ipynb`. Nothing here imports from
or depends on that notebook -- run this file top to bottom on its own.

Same evaluation protocol as `code_v1.ipynb` and `DeepRetro/notebooks/prod_challenge.ipynb`:
canonicalize predicted precursors and ground-truth reactants with RDKit, then require equal
cardinality plus set membership. `all_correct` means every predicted precursor is in the
ground truth with matching counts; `any_correct` means at least one is.

## Why vLLM

The HuggingFace `model.generate()` loop in `code_v1.ipynb` runs at **~8.1 s/molecule**
(measured: 250 rows in 33.9 min). Three things cost that time, none of them the model:

1. `BATCH_SIZE = 4`, sized for the 2xT4 Kaggle box the original comments describe. This
   machine is a single RTX PRO 6000 Blackwell (96GB, sm_120); a 7B model in bf16 is ~15GB, so
   the card sits almost entirely idle.
2. **Ragged batches.** Generated lengths run mean 578 / p90 754 / max 1024 tokens, and
   `generate()` runs every batch until its *longest* member finishes -- about 1.77x (max/mean)
   of the decode steps are spent on sequences that already stopped.
3. **No prefix reuse.** `SYS_PROMPT_OPENAI` + `USER_PROMPT_OPENAI` is ~600 tokens and is
   byte-identical across all 250 rows apart from the embedded SMILES, yet it is recomputed
   from scratch on every call.

vLLM fixes all three: continuous batching, a paged KV cache, and automatic prefix caching.
Measured result: **1.12 min for 250 molecules (0.27 s/mol), a 30.1x speedup**, with top-1
scores, parse-failure rate and mean proposal count all matching the HF engine.

## Kernel

This notebook needs the **Python (vLLM)** kernel, not the one `code_v1.ipynb` uses. vLLM pins
`torch==2.13` and `transformers>=5.5`, which cannot coexist with `/venv/main` (torch 2.10,
transformers 4.57), so it lives in its own venv:

```bash
python3 -m venv /venv/vllm
/venv/vllm/bin/pip install vllm==0.29.0 rdkit pandas accelerate ipywidgets ipykernel
/venv/vllm/bin/python -m ipykernel install --user --name vllm --display-name "Python (vLLM)"
```

Verify you are on it with `import sys, vllm; print(sys.executable, vllm.__version__)` ->
`/venv/vllm/bin/python 0.29.0`.

In [ ]:
from huggingface_hub import login
login()

In [13]:
import ast
import json
import os
import re
import sys
import time

import pandas as pd
import torch
from rdkit import Chem, RDLogger
from transformers import AutoTokenizer

RDLogger.DisableLog("rdApp.*")  # silence the parse errors we already handle

REPO = "/root/workspace/DFS/DeepRetro"
sys.path.insert(0, REPO)

# DeepRetro's own prompts. SYS/USER_PROMPT_OPENAI are the non-CoT pair the repo uses for
# models that do not emit <cot> tags -- the right family for instruct models like
# Olmo-3-Instruct. src/variables.py is pure string constants, so this pulls in no
# litellm/langfuse.
from src.variables import SYS_PROMPT_OPENAI, USER_PROMPT_OPENAI

# --- the only knob that has to change to evaluate a different model ---
MODEL_ID = "Qwen/Qwen3-8B"

DATA_CSV = f"{REPO}/data/uspto_50k_test_250.csv"
MAX_NEW_TOKENS = 1024

# Paths derive from MODEL_ID so two models never share a checkpoint file --
# run_generation_vllm() resumes from VLLM_JSONL, so a shared path would make a new model
# silently inherit the previous model's completions.
MODEL_SLUG = MODEL_ID.rstrip("/").split("/")[-1]
DATA_SLUG = os.path.splitext(os.path.basename(DATA_CSV))[0]
OUT_DIR = f"{REPO}/results/{DATA_SLUG}_{MODEL_SLUG}"

VLLM_JSONL = f"{OUT_DIR}/raw_generations_vllm.jsonl"   # this notebook writes here
RESULTS_CSV = f"{OUT_DIR}/results_vllm.csv"
# code_v1.ipynb's HF checkpoint. Read-only, and only for the optional comparison at the
# end -- this notebook never writes to it.
RAW_JSONL = f"{OUT_DIR}/raw_generations.jsonl"

os.makedirs(OUT_DIR, exist_ok=True)
print("writing to", OUT_DIR)

df_eval = pd.read_csv(DATA_CSV)
print(df_eval.shape, df_eval.columns.tolist())
df_eval.head()

writing to /root/workspace/DFS/DeepRetro/results/uspto_50k_test_250_Qwen3-8B
(250, 4) ['input', 'output', 'reaction_type', 'cluster_id']


,input,output,reaction_type,cluster_id
0,CS(=O)c1cccc(-c2nc(C=O)ccc2OCCO[Si](C)(C)C(C)(...,CC(C)(C)[Si](C)(C)OCCOc1ccc(C=O)nc1Br.CS(=O)c1...,3,0
1,CC(C)=CCSc1ccc(Br)cc1,CC(C)=CCBr.Sc1ccc(Br)cc1,1,0
2,CCC1(c2ccc(C=O)s2)OCCO1,CCC1(c2cccs2)OCCO1.CN(C)C=O,3,0
3,O=C1Nc2ccccc2C1c1cc(Br)ccc1O,O=C1Nc2ccccc2C1(O)c1cc(Br)ccc1O,9,0
4,CCCCCCCCCCCCCCCCOCC(CN)CC#N,CCCCCCCCCCCCCCCCOCC(CC#N)CN=[N+]=[N-],9,0


In [14]:
# Tokenizer only -- no AutoModelForCausalLM. vLLM loads its own copy of the weights, so
# nothing heavyweight is pulled into this process. The tokenizer is still needed because
# build_prompt() renders the chat template.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def build_prompt(molecule: str) -> str:
    """Render the chat prompt exactly as src/utils/llm.py::call_LLM builds its messages.

    Note: .replace() rather than .format() -- USER_PROMPT_OPENAI embeds a literal JSON
    schema with { } braces, so .format() would raise.

    Base models ship no chat template, so fall back to a plain system+user concatenation
    rather than crashing.
    """
    user = USER_PROMPT_OPENAI.replace("{target_smiles}", molecule)
    if getattr(tokenizer, "chat_template", None):
        messages = [
            {"role": "system", "content": SYS_PROMPT_OPENAI},
            {"role": "user", "content": user},
        ]
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    return f"{SYS_PROMPT_OPENAI}\n\n{user}\n\n"


print(build_prompt(df_eval.iloc[0]["input"])[:1200])

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

<|im_start|>system
You are an expert organic chemist specializing in retrosynthesis. When given a target molecule, you will perform a single-step retrosynthesis, providing 3-5 possible precursor molecules or reactions that could lead to the formation of the target molecule. 

Present your final analysis in a specific JSON format. For each suggestion, provide the precursor molecules in SMILES notation and a brief explanation of the reaction type and any key conditions or reagents needed. Use standard organic chemistry notation and terminology in your explanations. 

If the molecule is too simple for meaningful retrosynthesis, state this in a single JSON object with an appropriate explanation.
<|im_end|>
<|im_start|>user
You are an expert organic chemist specializing in retrosynthesis. When given a target molecule, you will perform a single-step retrosynthesis, providing 3-5 possible precursor molecules or reactions that could lead to the formation of the target molecule. 

Present your 

In [7]:
# Ports of src/utils/llm.py::split_json_openAI and ::validate_split_json.
# Inlined rather than imported: src.utils.llm pulls in litellm/langfuse, which are not
# installed here. Two robustness additions over the originals, because a 7B model drops
# the <json> tags and emits real JSON far more often than Claude/o1 do:
#   - json.loads first, ast.literal_eval as fallback (the repo uses only the latter)
#   - regex fallback to a bare {...} block when the tags are missing

_BARE_JSON = re.compile(r"\{.*\"data\".*\}", re.DOTALL)


def split_json_content(res_text: str):
    """Return (status, json_content). 200 on success, 502 on failure."""
    start = res_text.find("<json>")
    end = res_text.find("</json>")
    if start != -1 and end != -1 and end > start:
        content = res_text[start + len("<json>"):end].strip()
        if content:
            return 200, content
    m = _BARE_JSON.search(res_text)  # tags missing -- salvage the object itself
    if m:
        return 200, m.group(0).strip()
    return 502, ""


def loads_lenient(json_content: str):
    try:
        return json.loads(json_content)
    except Exception:
        return ast.literal_eval(json_content)  # Python-literal style (repo behaviour)


def validate_split_json(json_content: str):
    """Return (status, molecules, explanations, confidence_scores). 504 on failure."""
    try:
        result = loads_lenient(json_content)
        return 200, result["data"], result["explanation"], result["confidence_scores"]
    except Exception:
        return 504, [], [], []


def parse_response(res_text: str):
    """Full response -> (status, proposals). proposals is a list of precursor-SMILES lists."""
    status, json_content = split_json_content(res_text)
    if status != 200:
        return status, []
    status, molecules, _expl, _conf = validate_split_json(json_content)
    if status != 200:
        return status, []
    # Normalise: a single flat list of strings means one proposal, not many.
    if molecules and all(isinstance(m, str) for m in molecules):
        molecules = [molecules]
    proposals = [
        [s for s in prop if isinstance(s, str) and s.strip()]
        for prop in molecules
        if isinstance(prop, list)
    ]
    proposals = [p for p in proposals if p]
    return (200, proposals) if proposals else (504, [])

In [5]:
def load_checkpoint(path):
    done = {}
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line:
                    rec = json.loads(line)
                    done[rec["mol_no"]] = rec
    return done

In [8]:
def canon(smiles: str):
    try:
        return Chem.CanonSmiles(smiles)
    except Exception:
        return None


def score_proposal(pred_smiles_list, gt_canon):
    """Return (any_correct, all_correct, not_common, missed, valid)."""
    pred = [canon(s) for s in pred_smiles_list]
    if any(p is None for p in pred):
        return 0, 0, [s for s, p in zip(pred_smiles_list, pred) if p is None], gt_canon, False
    same_len = len(gt_canon) == len(pred)
    any_c = int(any(p in gt_canon for p in pred) and same_len)
    all_c = int(all(p in gt_canon for p in pred) and same_len)
    not_common = [p for p in pred if p not in gt_canon]
    missed = [g for g in gt_canon if g not in pred]
    return any_c, all_c, not_common, missed, True


def score_record(rec):
    """Score one generation record at top-1 and top-k."""
    gt_canon = [canon(s) for s in str(rec["output"]).split(".")]
    gt_canon = [g for g in gt_canon if g is not None]

    status, proposals = parse_response(rec["raw"])
    row = {
        "mol_no": rec["mol_no"],
        "input": rec["input"],
        "output": rec["output"],
        "parse_status": status,
        "n_proposals": len(proposals),
        "proposals": proposals,
        "any_correct": 0,
        "all_correct": 0,
        "any_correct_topk": 0,
        "all_correct_topk": 0,
        "first_hit_rank": None,
        "not_common": [],
        "missed": gt_canon,
        "has_invalid_smiles": 0,
    }
    if status != 200 or not proposals:
        return row  # parse failure counts as incorrect, it is not dropped

    any_invalid = False
    for rank, prop in enumerate(proposals, start=1):
        any_c, all_c, not_common, missed, valid = score_proposal(prop, gt_canon)
        if not valid:
            any_invalid = True
        if rank == 1:  # top-1: exactly what prod_challenge scores
            row.update(any_correct=any_c, all_correct=all_c,
                       not_common=not_common, missed=missed)
        row["any_correct_topk"] = max(row["any_correct_topk"], any_c)
        if all_c and row["first_hit_rank"] is None:
            row["first_hit_rank"] = rank
        row["all_correct_topk"] = max(row["all_correct_topk"], all_c)
    row["has_invalid_smiles"] = int(any_invalid)
    return row

# Sanity check: the ground truth must score (1, 1) against itself on every row.
_gt_ok = all(
    score_proposal([s for s in str(r["output"]).split(".")],
                   [canon(s) for s in str(r["output"]).split(".")])[:2] == (1, 1)
    for _, r in df_eval.iterrows()
)
print("ground truth scores perfectly against itself:", _gt_ok)

ground truth scores perfectly against itself: True


In [ ]:
# flashinfer JIT-compiles its sampling kernels on first use, and that build fails on this
# box (sm_120): curand.h is not on the include path, and forcing it there then hits a
# bundled-cccl header mismatch. We decode greedily, so those kernels buy us nothing --
# switch them off and use vLLM's PyTorch sampler. Must be set before vllm is imported.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

from vllm import LLM, SamplingParams

# Sized from what is actually free, so this works even if another kernel still holds GPU
# memory. gpu_memory_utilization is a fraction of *total*, not of free.
_free, _total = torch.cuda.mem_get_info()
GPU_UTIL = round(min(0.85, (_free / _total) * 0.90), 2)
print(f"free {_free/1e9:.0f}GB / {_total/1e9:.0f}GB -> gpu_memory_utilization={GPU_UTIL}")

llm = LLM(
    model=MODEL_ID,
    dtype="bfloat16",
    max_model_len=2048,           # ~600 prompt + 1024 new, with headroom
    gpu_memory_utilization=GPU_UTIL,
    enable_prefix_caching=True,   # the ~600-token shared preamble, computed once
)

In [ ]:
# Greedy, same token budget as the HF loop this replaces, so the two are comparable.
SAMPLING = SamplingParams(
    temperature=0.0,                  # greedy, matching the repo's temperature=0.0
    max_tokens=MAX_NEW_TOKENS,
    stop=["</json>"],                 # stop as soon as the object closes, rather than
    include_stop_str_in_output=True,  # rambling on to the 1024 cap; parse_response()
)                                     # looks for the closing tag, so it is kept.


def run_generation_vllm(df, limit=None):
    """Generate completions for df, appending to VLLM_JSONL and skipping finished rows.

    No BATCH_SIZE: vLLM schedules continuously, so one call covers the whole set and a
    finished sequence leaves the batch instead of waiting for its slowest neighbour.
    """
    done = load_checkpoint(VLLM_JSONL)
    todo = [(i, row) for i, row in df.iterrows() if int(i) not in done]
    if limit is not None:
        todo = todo[:limit]
    print(f"{len(done)} already done, generating {len(todo)}")
    if not todo:
        return

    prompts = [build_prompt(row["input"]) for _, row in todo]
    t0 = time.time()
    outs = llm.generate(prompts, SAMPLING)
    elapsed = time.time() - t0

    # One write at the end rather than per batch -- acceptable when the whole run is
    # minutes rather than half an hour.
    with open(VLLM_JSONL, "a") as fout:
        for (i, row), out in zip(todo, outs):
            fout.write(json.dumps({
                "mol_no": int(i),
                "model_id": MODEL_ID,
                "input": row["input"],
                "output": row["output"],
                "raw": out.outputs[0].text,
            }) + "\n")
        fout.flush()

    gen_toks = sum(len(o.outputs[0].token_ids) for o in outs)
    print(f"{len(todo)} molecules in {elapsed/60:.2f} min "
          f"({elapsed/len(todo):.2f} s/mol, {gen_toks/elapsed:.0f} tok/s)")

In [ ]:
# Smoke test: one molecule end to end, through the same parser the scoring uses.
_smoke = llm.generate([build_prompt(df_eval.iloc[0]["input"])], SAMPLING)[0].outputs[0].text
print(_smoke[:2000])
print("\n--- parsed ---")
print(parse_response(_smoke))

In [ ]:
# Dry run on the first 10. Re-running this cell should say "10 already done, generating 0"
# -- that confirms the checkpoint works before committing to the full set.
run_generation_vllm(df_eval, limit=10)

In [ ]:
# Full run over all 250. ~1 min, against ~34 min for the HF loop. Resumable, so it is safe
# to interrupt and re-run.
run_generation_vllm(df_eval)

In [ ]:
# Score every vLLM row, then -- if code_v1.ipynb's HF checkpoint happens to be present --
# put the two engines side by side over the identical set of molecules. That comparison is
# the regression check: a large gap means a prompt or stop-token problem, not a real change
# in model accuracy. The notebook stands alone without it.
recs_vllm = load_checkpoint(VLLM_JSONL)
df_vllm = pd.DataFrame([score_record(r) for r in recs_vllm.values()]).set_index("mol_no")
df_vllm = df_vllm.sort_index()
df_vllm.to_csv(RESULTS_CSV)

n = len(df_vllm)
print(f"model: {MODEL_ID}")
print(f"scored {n} / {len(df_eval)} molecules from {VLLM_JSONL}\n")

print("=== top-1 (first proposal -- prod_challenge protocol) ===")
print("All correct %", 100 * df_vllm.all_correct.sum() / n)
print("Any correct %", 100 * df_vllm.any_correct.sum() / n)

print("\n=== top-k (best of the 3-5 proposals) ===")
print("All correct %", 100 * df_vllm.all_correct_topk.sum() / n)
print("Any correct %", 100 * df_vllm.any_correct_topk.sum() / n)

print("\n=== response quality ===")
print(f"parse failures  : {100 * (df_vllm.parse_status != 200).sum() / n:.2f}%")
print(f"invalid SMILES  : {100 * df_vllm.has_invalid_smiles.sum() / n:.2f}%")
print(f"mean proposals  : {df_vllm.n_proposals.mean():.2f}")
print(f"\nDenominator is all {n} scored rows -- parse failures count as incorrect.")


def _summary(d):
    return [100 * d.all_correct.mean(), 100 * d.any_correct.mean(),
            100 * d.all_correct_topk.mean(), 100 * d.any_correct_topk.mean(),
            100 * (d.parse_status != 200).mean(), d.n_proposals.mean()]


comparison = None
if os.path.exists(RAW_JSONL):
    recs_hf = load_checkpoint(RAW_JSONL)
    common = sorted(set(recs_vllm) & set(recs_hf))
    if common:
        df_v = pd.DataFrame([score_record(recs_vllm[i]) for i in common]).set_index("mol_no")
        df_h = pd.DataFrame([score_record(recs_hf[i]) for i in common]).set_index("mol_no")
        comparison = pd.DataFrame(
            {"vLLM": _summary(df_v), "HF (batch=4)": _summary(df_h)},
            index=["top-1 all_correct %", "top-1 any_correct %",
                   "top-k all_correct %", "top-k any_correct %",
                   "parse failures %", "mean proposals"],
        ).round(2)
        print(f"\ncomparing {len(common)} molecules present in both runs")
else:
    print(f"\nno HF run at {RAW_JSONL} -- skipping engine comparison")

comparison if comparison is not None else df_vllm.head(10)